## MLP con múltiples salidas

Iris es el género de una planta herbácea con flores que se utilizan en decoración. Dentro de este género existen muy diversas especies, entre las que se han estudiado: Iris setosa, Iris versicolor e Iris virginica.
Estas tres especies pueden distinguirse según las dimensiones de sus pétalos y sépalos. Un grupo de investigadores ha recopilado la información correspondiente a las longitudes y anchos de los pétalos y sépalos de 50 plantas de cada especie. 
En el archivo iris81 trn.csv se encuentra el conjunto de entrenamiento, y en iris81 tst.csv el de prueba, generado a partir de estas mediciones (en cm), junto con un código binario que indica la clase de cada muestra (especie) reconocida por el grupo de investigadores:
([−1, −1, 1] = setosa,
[−1, 1, −1] = versicolor,
[1, −1, −1] = virginica)


### Inicialización


In [1]:
import numpy as np
import pandas as pd
import random as random
import copy


class Capa:
    w : np.ndarray
    y: np.ndarray
    delta: np.ndarray

    def __init__(self, w_i, y_i, delta_i):
        self.w = w_i
        self.y = y_i
        self.delta = delta_i

    def mostrar(self):
        print(f"Pesos: {self.w}")
        print(f"Salidas: {self.y}")
        print(f"Deltas: {self.delta}\n")


def sigm(x):
    return (2/(1+np.exp(-x))) - 1

entrada_usuario = [2,3] 
# Primer prueba: 2 capas: 4 neuronas capa oculta, 3 neuronas capa salida (para encajar con la estructura de una salida esperada) ---> converge (~50-60 epocas)
# Segunda prueba: 3 capas: [4, 2, 3] ---> converge (70-90 epocas)
# Tercer prueba: 4 capas: [4, 3, 2, 3] ---> converge (menos veces) (80-110 epocas)

tabla = pd.read_csv('../../Data/gtp_2/iris81_trn.csv', header=None).to_numpy()
x0 = -np.ones(len(tabla))
entradas = np.c_[x0, tabla[:,:4]] # extraer las primeras 4 columnas
yd_vect =  tabla[:, 4:] # extraer desde la quinta hasta el final

print(entradas.shape)
print(yd_vect.shape)

w = np.random.rand(entrada_usuario[0], len(entradas[0])) - 0.5
y_init = np.zeros(entrada_usuario[0])
delta = np.zeros(entrada_usuario[0])
cap = Capa(w,y_init,delta)
vect_capas = [copy.deepcopy(cap)]

# Iniciar red (aleatorio):
for i in range(1,len(entrada_usuario)):
    w = np.random.rand(entrada_usuario[i], entrada_usuario[i-1]+1) - 0.5
    y_init = np.zeros(entrada_usuario[i])
    delta = np.zeros(entrada_usuario[i])
    cap = Capa(w,y_init,delta)
    vect_capas.append(copy.deepcopy(cap))


# Visualizar red inicial:
print(f"Cantidad de capas: {len(vect_capas)}\n")
print(f"Red neuronal: \n")
i = 1
for capa in vect_capas:
    print(f"Capa: {i}")
    capa.mostrar()
    i += 1


(111, 5)
(111, 3)
Cantidad de capas: 2

Red neuronal: 

Capa: 1
Pesos: [[-0.06431897  0.07647381 -0.21586965  0.37932042 -0.19699499]
 [-0.06746803  0.14282938 -0.38609929 -0.06218103  0.3861152 ]]
Salidas: [0. 0.]
Deltas: [0. 0.]

Capa: 2
Pesos: [[-0.02111531 -0.38589861 -0.05115638]
 [ 0.16473878  0.00250359 -0.36487619]
 [ 0.32049284  0.2483986  -0.14295702]]
Salidas: [0. 0. 0.]
Deltas: [0. 0. 0.]



### Entrenamiento


In [2]:
# Iterar sobre la red:

epoca = 1
epoca_max = 500
mu = 0.1

# n -> ejemplo actual
# i -> la capa
# j -> la neurona
while epoca < epoca_max: 
    # SOLUCIÓN: Iterar sobre len(entradas) (las filas), no len(entradas[0]) (las columnas)
    for n in range(len(entradas)):

        # paso hacia adelante
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                if i==0:
                    z = np.dot(entradas[n,:],vect_capas[i].w[j,:])
                else: 
                    ent = np.r_[-1,vect_capas[i-1].y]
                    z = np.dot(ent,vect_capas[i].w[j,:])
                vect_capas[i].y[j] = sigm(z)

        # propagacion hacia atras
        # SOLUCIÓN: Empezar en len(vect_capas)-1 para no tener un IndexError
        for i in range(len(vect_capas)-1,-1,-1):
            for j in range(len(vect_capas[i].y)):
                
                if i==len(vect_capas)-1: # Capa de salida
                    # el deseado para el ejemplo 'n' es un vector ---> recorrer cada columna 'j', asi coincide con el tamaño de vect_capas[i].y
                    vect_capas[i].delta[j] = (1/2) * (yd_vect[n,j] - vect_capas[i].y[j]) * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])
                else:
                    vect_capas[i].delta[j] = (1/2) * np.dot(vect_capas[i+1].delta, vect_capas[i+1].w[:,j+1])  * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])

        #actualizar los pesos
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                # SOLUCIÓN: Usar len de los pesos de la neurona actual
                for m in range(len(vect_capas[i].w[j])):
                    if i==0:
                        # SOLUCIÓN: Usar += en lugar de -=
                        vect_capas[i].w[j,m] += mu*vect_capas[i].delta[j]*entradas[n,m]
                    else:
                        # SOLUCIÓN: Hay que reconstruir la entrada con el -1 para poder usar el índice 'm' sin salir de rango, y usar +=
                        ent = np.r_[-1, vect_capas[i-1].y]
                        vect_capas[i].w[j,m] += mu*vect_capas[i].delta[j]*ent[m]
                        
    # Verificación:
    acierto = 0
    for n in range(len(entradas)):
        for capa in range(len(vect_capas)):
            for neuron in range(len(vect_capas[capa].y)):
                if capa==0:
                    z = np.dot(entradas[n,:],vect_capas[capa].w[neuron,:])
                else:                
                    ent = np.r_[-1,vect_capas[capa-1].y]
                    z = np.dot(ent,vect_capas[capa].w[neuron,:])

                vect_capas[capa].y[neuron] = sigm(z)

        salida_red = vect_capas[-1].y # la salida es un vector de 3
        # salida_red[salida_red <= 0] = -1 # todos los valores 'bajos' = -1
        # salida_red[salida_red > 0] = 1 # el valor que tiene que ser 1 (maximo) = 1

        idx_max = np.argmax(salida_red)
        for idx in range(len(salida_red)):
            if idx == idx_max:
                salida_red[idx] = 1
            else:
                salida_red[idx] = -1

        if ((salida_red == yd_vect[n,:]).all()): # si salida_red coincide exactamente con el deseado para el ejemplo 'n' --> acierto
            acierto += 1

    tasa_acierto = acierto/len(entradas)
    print(f"Fin entrenamiento. Epoca: {epoca}, Tasa de acierto: {tasa_acierto * 100: .2f}\n")

    if tasa_acierto >= 0.99:
        print(f"Convergencia. Tasa de aciertos: {tasa_acierto}, Epoca: {epoca}\n")
        break
    else:
        epoca += 1

print(f"Red neuronal final: \n")
i = 1
for cap in vect_capas:
    print(f"Capa {i}:")
    cap.mostrar()
    i += 1



Fin entrenamiento. Epoca: 1, Tasa de acierto:  85.59

Fin entrenamiento. Epoca: 2, Tasa de acierto:  99.10

Convergencia. Tasa de aciertos: 0.990990990990991, Epoca: 2

Red neuronal final: 

Capa 1:
Pesos: [[-0.17429108  0.6268638   0.17001415  0.48494918 -0.191331  ]
 [ 0.14218411 -0.30248807 -1.21179453  1.18519911  0.95087492]]
Salidas: [ 0.98126489 -0.92117933]
Deltas: [ 0.0007636  -0.02236529]

Capa 2:
Pesos: [[ 0.49539692 -0.71710076  1.43058476]
 [ 0.58143395 -0.24627233  0.79540533]
 [ 0.32089432  0.01991778 -2.33038033]]
Salidas: [-1. -1.  1.]
Deltas: [-0.02125192 -0.10563474  0.06805309]



### Test: iris_tst


In [3]:
tabla_tst = pd.read_csv('../../Data/gtp_2/iris81_tst.csv', header=None).to_numpy()

x0_tst = -np.ones(len(tabla_tst))
entradas_tst = np.c_[x0_tst, tabla_tst[:,:4]] 
yd_tst =  tabla_tst[:, 4:]

acierto = 0
for n in range(len(entradas_tst)):
    for capa in range(len(vect_capas)):
        for neuron in range(len(vect_capas[capa].y)):
            if capa==0:
                z = np.dot(entradas_tst[n,:],vect_capas[capa].w[neuron,:])
            else:                
                ent = np.r_[-1,vect_capas[capa-1].y]
                z = np.dot(ent,vect_capas[capa].w[neuron,:])

            vect_capas[capa].y[neuron] = sigm(z)

    salida_red_tst = vect_capas[-1].y # la salida es un vector de 3
    idx_max = np.argmax(salida_red_tst)
    for idx in range(len(salida_red_tst)):
        if idx == idx_max:
            salida_red_tst[idx] = 1
        else:
            salida_red_tst[idx] = -1

    if ((salida_red_tst == yd_tst[n,:]).all()): # si salida_red coincide exactamente con el deseado para el ejemplo 'n' --> acierto
        acierto += 1

tasa_acierto = acierto/len(entradas_tst)
print(f"Test: Tasa de acierto: {tasa_acierto * 100: .2f}%")


Test: Tasa de acierto:  91.89%


### Parte 2

Explore cómo varı́a el desempeño al usar distintas tasas de aprendizaje, y para cada caso grafique las curvas de error cuadrático total y error de clasificación en función de las épocas de entrenamiento.

In [4]:
%matplotlib QtAgg
import matplotlib.pyplot as plt

# Iterar sobre la red:

epoca = 1
epoca_max = 500
mu = [0.2, 0.01, 0.5, 0.45, 0.25] 
k = 0 # variable para pasar de mu

# diccionarios para los errores de todas las epocas de cada tasa de aprendizaje
error_cuad_h = {}
error_clas_h = {}

# n -> ejemplo actual
# i -> la capa
# j -> la neurona
while k < len(mu):
    # volver a inicializar la red
    epoca = 1
    # Al fijar la semilla al inicio del ciclo, la secuencia de números generada a continuación será EXACTAMENTE la misma para k=0, k=1 y k=2, etc.
    np.random.seed(40)

    # errores para cada epoca
    error_cuad_e = []
    error_clas_e = []

    w = np.random.rand(entrada_usuario[0], len(entradas[0])) - 0.5
    y_init = np.zeros(entrada_usuario[0])
    delta = np.zeros(entrada_usuario[0])
    cap = Capa(w,y_init,delta)
    vect_capas = [copy.deepcopy(cap)]

    # Iniciar red (aleatorio):
    for i in range(1,len(entrada_usuario)):
        w = np.random.rand(entrada_usuario[i], entrada_usuario[i-1]+1) - 0.5
        y_init = np.zeros(entrada_usuario[i])
        delta = np.zeros(entrada_usuario[i])
        cap = Capa(w,y_init,delta)
        vect_capas.append(copy.deepcopy(cap))

    # volver a iterar
    while epoca < epoca_max: 

        for n in range(len(entradas)):

            # paso hacia adelante
            for i in range(len(vect_capas)):
                for j in range(len(vect_capas[i].y)):
                    if i==0:
                        z = np.dot(entradas[n,:],vect_capas[i].w[j,:])
                    else: 
                        ent = np.r_[-1,vect_capas[i-1].y]
                        z = np.dot(ent,vect_capas[i].w[j,:])
                    vect_capas[i].y[j] = sigm(z)

            # propagacion hacia atras
            # SOLUCIÓN: Empezar en len(vect_capas)-1 para no tener un IndexError
            for i in range(len(vect_capas)-1,-1,-1):
                for j in range(len(vect_capas[i].y)):
                    
                    if i==len(vect_capas)-1: # Capa de salida
                        # el deseado para el ejemplo 'n' es un vector ---> recorrer cada columna 'j', asi coincide con el tamaño de vect_capas[i].y
                        vect_capas[i].delta[j] = (1/2) * (yd_vect[n,j] - vect_capas[i].y[j]) * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])
                    else:
                        vect_capas[i].delta[j] = (1/2) * np.dot(vect_capas[i+1].delta, vect_capas[i+1].w[:,j+1])  * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])

            #actualizar los pesos
            for i in range(len(vect_capas)):
                for j in range(len(vect_capas[i].y)):
                    # SOLUCIÓN: Usar len de los pesos de la neurona actual
                    for m in range(len(vect_capas[i].w[j])):
                        if i==0:
                            # SOLUCIÓN: Usar += en lugar de -=
                            vect_capas[i].w[j,m] += mu[k]*vect_capas[i].delta[j]*entradas[n,m]
                        else:
                            # SOLUCIÓN: Hay que reconstruir la entrada con el -1 para poder usar el índice 'm' sin salir de rango, y usar +=
                            ent = np.r_[-1, vect_capas[i-1].y]
                            vect_capas[i].w[j,m] += mu[k]*vect_capas[i].delta[j]*ent[m]
                            
        # Verificación:
        acierto = 0
        error_cuad = 0
        for n in range(len(entradas)):
            for capa in range(len(vect_capas)):
                for neuron in range(len(vect_capas[capa].y)):
                    if capa==0:
                        z = np.dot(entradas[n,:],vect_capas[capa].w[neuron,:])
                    else:                
                        ent = np.r_[-1,vect_capas[capa-1].y]
                        z = np.dot(ent,vect_capas[capa].w[neuron,:])

                    vect_capas[capa].y[neuron] = sigm(z)

            salida_red = vect_capas[-1].y # la salida es un vector de 3

            error_cuad += np.sum((yd_vect[n,:] - salida_red)**2) # '**' equivale a potencia
            
            idx_max = np.argmax(salida_red)
            for idx in range(len(salida_red)):
                if idx == idx_max:
                    salida_red[idx] = 1
                else:
                    salida_red[idx] = -1

            if ((salida_red == yd_vect[n,:]).all()): # si salida_red coincide exactamente con el deseado para el ejemplo 'n' --> acierto
                acierto += 1

        tasa_acierto = acierto/len(entradas)
        error_clasificacion = 1 - tasa_acierto # el error es el complemento del acierto

        error_cuad_e.append(error_cuad)
        error_clas_e.append(error_clasificacion)

        if tasa_acierto > 0.98:
            print(f"Convergencia. Tasa de aciertos: {tasa_acierto}, Epoca: {epoca}, Tasa aprendizaje: mu = {mu[k]}\n")
            break
        else:
            epoca += 1

        
    if epoca == epoca_max:
        print(f"No hubo convergencia ({epoca_max} epocas) - {tasa_acierto * 100: .2f}%. Tasa aprendizaje: mu = {mu[k]}\n")

    # guardar para la tasa 'k' los errores que obtuvo
    error_cuad_h[mu[k]] = error_cuad_e
    error_clas_h[mu[k]] = error_clas_e

    k += 1
    if k < len(mu):
        print("Pasando a tasa nueva...\n")
    else:
        break


# graficar
plt.figure(1,figsize=(10, 5))
for tasa, errores in error_cuad_h.items():
    plt.plot(errores, label=f'mu = {tasa}')

plt.title('Error Cuadrático por Epoca')
plt.xlabel('Epoca')
plt.ylabel('Error Cuadrático')
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend()
plt.show()

plt.figure(2,figsize=(10, 5))
for tasa, errores in error_clas_h.items():
    plt.plot(errores, label=f'mu = {tasa}')

plt.title('Error de Clasificación por Epoca')
plt.xlabel('Epoca')
plt.ylabel('Tasa de Error')
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend()
plt.show()


Convergencia. Tasa de aciertos: 0.990990990990991, Epoca: 38, Tasa aprendizaje: mu = 0.2

Pasando a tasa nueva...

Convergencia. Tasa de aciertos: 0.990990990990991, Epoca: 60, Tasa aprendizaje: mu = 0.01

Pasando a tasa nueva...

No hubo convergencia (500 epocas) -  40.54%. Tasa aprendizaje: mu = 0.5

Pasando a tasa nueva...

No hubo convergencia (500 epocas) -  69.37%. Tasa aprendizaje: mu = 0.45

Pasando a tasa nueva...

Convergencia. Tasa de aciertos: 0.990990990990991, Epoca: 218, Tasa aprendizaje: mu = 0.25




(python:7258): Gtk-WARNING **: 11:31:36.694: Could not load a pixbuf from icon theme.
This may indicate that pixbuf loaders or the mime database could not be found.
